In [1]:
%%capture
# 필요한 패키지 설치
%pip install "langchain>=0.3.0" "langchain-openai>=0.2.0" "langchain-upstage>=0.3.0" "langgraph>=0.2.0" "python-dotenv>=1.0.0" -q

In [2]:
import os
import warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# .env 파일에서 환경변수 로드
load_dotenv()

# API 키 확인 (.env 미설정 시 직접 입력)
# if not os.environ.get("OPENAI_API_KEY"):
#     os.environ["OPENAI_API_KEY"] = input("🔑 OPENAI_API_KEY를 입력하세요: ")

if not os.environ.get("UPSTAGE_API_KEY"):
    os.environ["UPSTAGE_API_KEY"] = input("🔑 UPSTAGE_API_KEY를 입력하세요: ")
print("✅ 환경 설정 완료")

✅ 환경 설정 완료


In [3]:
from langchain_openai import ChatOpenAI

# ═══════════════════════════════════════════════════════════════
# 모델 초기화 — OpenAI / Solar 중 택 1 (주석 해제하여 전환)
# ═══════════════════════════════════════════════════════════════

# ── Option A: OpenAI (기본) ──
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ── Option B: Solar 대안 (주석 해제하여 사용) ──
from langchain_upstage import ChatUpstage
llm = ChatUpstage(model="solar-pro3")

In [4]:
# 모델 연결 테스트
response = llm.invoke("3일 서울 여행 계획을 간단히 요약해줘.")
print("LLM 응답:")
print(response.content[:200] + "...")
print(f"\n✅ 모델 연결 확인 완료 ({llm.model_name})")

LLM 응답:
**3일 서울 여행 요약 (예시)**  

| 일차 | 주요 일정 | 비고 |
|------|----------|------|
| **1일차** | - **인천공항 → 명동** (도착 후 점심) <br> - 명동 쇼핑·길거리 음식 <br> - **남산타워** (전망대·케이블카) <br> - 저녁: 남영동·용산 일대 야경·맛집 | 낮·밤 모두 활기찬 분위기 ...

✅ 모델 연결 확인 완료 (solar-pro3)


In [5]:
# TODO: 단일 Agent로 복잡한 작업 시도

# 복잡한 작업 요청
complex_request = """
3일 서울 여행 계획을 세워줘.
- 각 날짜별 상세 일정
- 점심, 저녁 맛집 추천 (위치, 메뉴, 가격대)
- 관광지 추천 (입장료, 소요시간)
- 전체 예산 계산
"""

# TODO: llm.invoke()를 사용하여 단일 Agent로 처리하세요
response = llm.invoke(complex_request)

if response is not None:
    print("=== 단일 Agent 응답 ===")
    print(response.content)
else:
    print("=== 코드를 완성해주세요 ===")

=== 단일 Agent 응답 ===
서울 3일 여행 계획

---

## 📅 전체 개요
| 항목 | 내용 |
|------|------|
| **여행 기간** | 2026‑03‑28 (월) ~ 2026‑03‑30 (수) |
| **숙소** | **롯데시티호텔 서울** (잠실역 바로 앞, 4성급) <br>※ 교통, 쇼핑, 엔터테인먼트 모두 편리 |
| **교통** | 교통카드 (T‑money) 1일 패스: 8,000원 <br>※ 도보, 지하철, 버스 이용 중심 (택시 최소화) |
| **예산** | **총 예상 비용** 1,850,000원 (한화) |
| **일일 예산** | 숙박 280,000원 (3박) → **하루 숙박 ≈ 93,000원** <br>식사·교통·입장료 포함 **하루 55,000원** |

---

## 🗓️ 1일차 (월요일) – 강남·코엑스·한강·잠실

| 시간 | 일정 | 상세 내용 | 비용 |
|------|------|-----------|------|
| 08:30 | 숙소 체크‑인 | 롯데시티호텔 | 93,000원 (숙박) |
| 09:30 | **코엑스 몰** | - **아쿠아리움** (코엑스 아쿠아리움) <br>입장료: 22,000원 <br>소요: 1.5시간 <br>- **메가박스** 영화 관람 (점심 전) | 22,000원 |
| 12:00 | 점심 – **스시 하코** | 위치: 삼성동, 코엑스 내부 <br>베스트 메뉴: <br>• 사시미 세트 (15,000원) <br>• 마끼 (10,000원) <br>가격대: 1인당 12,000~18,000원 | 35,000원 (2인) |
| 14:30 | **현대백화점 무역센터점** 쇼핑 | 면세점, 패션, 화장품 등 | 0원 (선택) |
| 16:00 | **잠실 롯데월드** (테마파크) | 입장료: 45,000원 (성인) <br>소요: 3시간 <br>※ 저녁까지 놀이기구 이용 | 45,000원 |
| 19:00 | 저녁 – **치맥**: **치맥펍 (Coex Beer & Grill)** 

# Multi-Agent

In [6]:
# LangGraph import 및 State 정의
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

class AgentState(TypedDict):
    """Multi-Agent 시스템의 상태 정의"""
    user_request: str     # 사용자 요청
    plan: str             # Planner가 생성한 계획
    result: str           # Worker가 생성한 결과
    reflection: str       # Reflection Agent의 검토 결과
    final_result: str     # 최종 결과
    reflection_count: int # Reflection 반복 횟수 (무한 루프 방지)
    messages: List[dict]  # 에이전트 간 메시지 기록

# State 구조 확인
print("AgentState 필드:")
print("  - user_request: 사용자 요청 문자열")
print("  - plan: Planner Agent의 계획")
print("  - result: Worker Agent의 실행 결과")
print("  - reflection: Reflection Agent의 검토 결과")
print("  - final_result: 최종 결과")
print("  - reflection_count: Reflection 반복 횟수 (무한 루프 방지)")
print("  - messages: 에이전트 간 메시지 기록")

AgentState 필드:
  - user_request: 사용자 요청 문자열
  - plan: Planner Agent의 계획
  - result: Worker Agent의 실행 결과
  - reflection: Reflection Agent의 검토 결과
  - final_result: 최종 결과
  - reflection_count: Reflection 반복 횟수 (무한 루프 방지)
  - messages: 에이전트 간 메시지 기록


## Planner - Worker

In [10]:
# TODO: Planner Node 구현

def planner_node(state: AgentState) -> AgentState:
    """
    Planner Node: 사용자 요청을 분석하고 단계별 계획을 수립합니다.
    
    Args:
        state: 현재 AgentState
    
    Returns:
        업데이트된 AgentState (plan 필드 추가)
    """
    user_request = state["user_request"]
    
    # TODO: Planner 프롬프트를 작성하세요
    # - 사용자 요청을 분석하고 단계별 계획을 수립하도록 지시
    # - "계획만 수립하고, 직접 실행하지 말 것"을 명시
    # - 출력 형식: Step 1: [작업], Step 2: [작업], ...
    planner_prompt = f"""
    당신은 작업 계획을 수립하는 Planner Agent입니다.

    사용자 요청을 분석하고, 수행해야 할 작업을 단계별로 나열하세요.
    각 단계는 구체적이고 실행 가능해야 합니다.
    
    중요: 계획만 수립하세요. 직접 실행하지 마세요.
    
    출력 형식:
    Step 1: [작업 설명]
    Step 2: [작업 설명]
    ...
    
    ---
    사용자 요청: {user_request}
    """
    
    # LLM 호출
    response = llm.invoke(planner_prompt)
    plan = response.content
    
    # 메시지 기록 추가
    messages = state.get("messages", [])
    messages.append({
        "sender": "Planner",
        "receiver": "Worker",
        "content": plan,
        "task_type": "plan"
    })
    
    # TODO: State 업데이트하여 반환
    return {
        **state,
        "plan": plan,  # TODO: plan 변수로 교체
        "messages": messages
    }

# 테스트
test_state = {
    "user_request": "3일 서울 여행 계획을 세워줘. 맛집, 관광지, 예산 포함.",
    "plan": "",
    "result": "",
    "messages": []
}

# TODO 완료 후 테스트
result_state = planner_node(test_state)
if result_state['plan'] is not None:
    print("=== Planner Node 결과 ===")
    print(f"계획:\n{result_state['plan']}")
else:
    print("=== 코드를 완성해주세요 ===")

=== Planner Node 결과 ===
계획:
Step 1: 사용자 요청 분석 및 핵심 요구사항 파악  
- 서울 3일 여행 계획 수립  
- 맛집, 관광지, 예산 포함  
- 단계별 실행 가능한 작업 목록 작성 (계획만 수립, 실행 금지)

Step 2: 여행 기간 및 주요 목표 정의  
- 3일간 방문 가능한 주요 관광지 선정  
- 맛집 방문 리스트 구성 (지역별, 코스별)  
- 예산 범위 설정 (일일 예산, 총 예산, 교통·숙박·식사 비용 등)

Step 3: 일정 초안 작성 (날짜별)  
- Day 1: 서울 도착 및 도심 탐방 (예: 인사동, 남산타워)  
- Day 2: 문화와 역사 탐방 (예: 경복궁, 북촌한옥마을)  
- Day 3: 쇼핑·현대 문화 체험 (예: 홍대, 강남)  

Step 4: 관광지 상세 정보 수집 및 선별  
- 각 관광지 운영 시간, 입장료, 예약 필요 여부 확인  
- 이동 거리 및 소요 시간 고려 (대중교통 중심)  

Step 5: 맛집 리스트 작성 및 상세 정보 수집  
- 지역별 대표 맛집 선정 (예: 강남, 명동, 홍대)  
- 메뉴, 가격대, 예약 가능 여부, 평점 정보 수집  

Step 6: 예산 산정 및 배분  
- 교통비 (지하철, 버스, 택시)  
- 숙박비 (호텔, 게스트하우스 등)  
- 식사비 (아침, 점심, 저녁)  
- 입장료 및 기타 비용 (기념품, 체험 활동 등)  

Step 7: 일정 최적화 및 조율  
- 관광지 간 이동 동선 최적화  
- 맛집 방문 시간 배분  
- 휴식 시간 및 여유 일정 확보  

Step 8: 최종 일정표 작성 및 요약  
- 날짜별 상세 일정 (시간, 장소, 활동)  
- 예산 항목별 내역 (총 예산, 일일 예산)  
- 추가 팁 (예: 할인 패스, 앱 추천)  

Step 9: 사용자 확인 및 피드백 대기  
- 초안 제공 후 사용자의 수정 요청 반영 준비  

Step 10: 최종 계획 확정 및 문서화  
- 확정된 일정을 표 형태로 정리  
- 

In [11]:
# TODO: Worker Node 구현

def worker_node(state: AgentState) -> AgentState:
    """
    Worker Node: Planner의 계획을 받아 각 단계를 실행합니다.
    
    Args:
        state: 현재 AgentState (plan 포함)
    
    Returns:
        업데이트된 AgentState (result 필드 추가)
    """
    plan = state["plan"]
    user_request = state["user_request"]
    
    # TODO: Worker 프롬프트를 작성하세요
    # - Planner의 계획을 받아 각 단계를 실행하도록 지시
    # - 각 단계별 구체적인 정보를 포함하도록 요청
    worker_prompt = f"""
    당신은 계획을 실행하는 Worker Agent입니다.

    Planner가 수립한 계획을 받아 각 단계를 실행하고 결과를 제공하세요.
    각 단계별로 구체적인 정보를 포함하세요.
    
    ---
    원본 요청: {user_request}
    
    실행할 계획:
    {plan}
    
    ---
    위 계획을 단계별로 실행하고, 각 단계의 결과를 자세히 작성하세요.
    """
    
    # LLM 호출
    response = llm.invoke(worker_prompt)
    result = response.content

    # 메시지 기록 추가
    messages = state.get("messages", [])
    messages.append({
        "sender": "Worker",
        "receiver": "User",
        "content": result,
        "task_type": "result"
    })
    
    # TODO: State 업데이트하여 반환
    return {
        **state,
        "result": result,  # TODO: result 변수로 교체
        "messages": messages
    }

# TODO 완료 후 테스트
final_state = worker_node(result_state)
if final_state['result'] is not None:
    print("=== Worker Node 결과 ===")
    print(f"실행 결과:\n{final_state['result'][:500]}...")
else:
    print("=== 코드를 완성해주세요 ===")

=== Worker Node 결과 ===
실행 결과:
We need to respond as a Worker Agent executing the plan. The user gave the plan steps to run: analysis, define goals, draft schedule, collect info, compile list, budget, optimize, final schedule, confirm, finalize. We need to execute each step and provide detailed results.

We need to produce step-by-step results, with concrete info. Since we don't have actual user preferences beyond generic, we can assume typical interests: culture, shopping, food, reasonable budget maybe mid-range. Provide pla...


In [12]:
from langgraph.graph import StateGraph, START, END

# LangGraph 워크플로우 정의 (Guided Build - 코드 제공)
workflow = StateGraph(AgentState)

# 노드 추가
workflow.add_node("planner", planner_node)
workflow.add_node("worker", worker_node)

# 엣지 연결: planner → worker → END
workflow.add_edge(START, "planner")
workflow.add_edge("planner", "worker")
workflow.add_edge("worker", END)

# 컴파일
app = workflow.compile()

print("LangGraph 워크플로우 구성 완료!")
print("흐름: planner → worker → END")

LangGraph 워크플로우 구성 완료!
흐름: planner → worker → END


In [14]:
# LangGraph 워크플로우 실행
def run_multi_agent(user_request: str) -> dict:
    """
    LangGraph Multi-Agent 시스템 실행
    
    Args:
        user_request: 사용자 요청
    
    Returns:
        최종 상태를 담은 딕셔너리
    """
    print("🚀 LangGraph Multi-Agent 시스템 시작\n")
    
    # 초기 상태 설정
    initial_state = {
        "user_request": user_request
    }
    
    # 워크플로우 실행
    print("📋 [Planner Agent] 계획 수립 중...")
    print("⚙️ [Worker Agent] 계획 실행 중...")
    
    final_state = app.invoke(initial_state)
    
    print("✅ 실행 완료\n")
    
    return final_state

# 테스트 실행
workflow_result = run_multi_agent("서울 2박3일 여행 계획 + 예산 50만원 이내")

print("=" * 50)
print("📊 최종 결과")
print("=" * 50)
print(workflow_result["result"])

🚀 LangGraph Multi-Agent 시스템 시작

📋 [Planner Agent] 계획 수립 중...
⚙️ [Worker Agent] 계획 실행 중...
✅ 실행 완료

📊 최종 결과
The user wants us to act as a worker agent executing the plan. We have to follow the steps, produce detailed execution results. We need to produce a final schedule and budget, presumably with concrete numbers, suggestions, and possibly a checklist. Must be in Korean. The original request is "서울 2박3일 여행 계획 + 예산 50만원 이내". The steps are enumerated. We need to execute each step, providing details. We need to produce a schedule, budget breakdown, recommendations, etc. Since we cannot actually book, we give user options and information. Provide detailed execution results for each step.

We'll produce a structured response with each step, results, tables, suggestions. Also include optional steps (checklist, emergency contacts). Ensure budget stays under 500,000 KRW. Provide approximate costs.

Let's do:

Step 1: basic framework - confirm.

Step 2: detailed activities with times.

Step 3:

## Reflection

In [15]:
# TODO: Reflection Node 구현

MAX_REFLECTIONS = 1  # 최대 Reflection 반복 횟수 (무한 루프 방지)

def reflection_node(state: AgentState) -> AgentState:
    """
    Reflection Node: Worker의 결과를 검토하고 개선 제안을 합니다.
    
    Args:
        state: 현재 AgentState (result 포함)
    
    Returns:
        업데이트된 AgentState (reflection, final_result 필드 추가)
    """
    result = state["result"]
    user_request = state["user_request"]
    plan = state["plan"]
    
    # TODO: Reflection 프롬프트를 작성하세요
    # - Worker의 결과를 검토하고 원본 요청 충족 여부 평가
    # - 검토 항목: 요구사항 충족, 누락 정보, 정확성, 개선 필요 사항
    # - 출력 형식: ## 검토 결과 + ## 최종 개선 결과
    reflection_prompt = f"""
    당신은 결과를 검토하는 Reflection Agent입니다.

    Worker Agent가 생성한 결과를 검토하고, 원본 요청을 충족하는지 평가하세요.
    
    검토 항목:
    1. 원본 요청의 모든 요구사항이 충족되었는가?
    2. 누락된 정보가 있는가?
    3. 정보의 정확성과 구체성은 적절한가?
    4. 개선이 필요한 부분이 있는가?
    
    ---
    원본 요청: {user_request}
    
    실행된 계획:
    {plan}
    
    Worker의 결과:
    {result}
    
    ---
    위 내용을 검토하고 다음 형식으로 작성하세요:
    
    ## 검토 결과
    - 충족 여부: [충족/부분 충족/미충족]
    - 강점: [잘된 부분]
    - 개선 필요: [부족한 부분]
    
    ## 최종 개선 결과
    [Worker의 결과를 보완하여 최종 결과 작성]

    """
    
    # LLM 호출
    response = llm.invoke(reflection_prompt)
    reflection = response.content

    # 메시지 기록 추가
    messages = state.get("messages", [])
    messages.append({
        "sender": "Reflection",
        "receiver": "User",
        "content": reflection,
        "task_type": "reflection"
    })
    
    # TODO: State 업데이트하여 반환 (reflection_count 증가 포함)
    return {
        **state,
        "reflection": reflection,  # TODO: reflection 변수로 교체
        "final_result": reflection,  # TODO: reflection 변수로 교체
        "reflection_count": state.get("reflection_count", 0) + 1,
        "messages": messages
    }

def should_continue_reflection(state: AgentState) -> str:
    """Reflection 후 종료할지 Worker로 돌아갈지 판단"""
    count = state.get("reflection_count", 0)
    reflection = state.get("reflection", "")
    
    # 최대 반복 횟수 초과 시 종료
    if count >= MAX_REFLECTIONS:
        return "end"
    
    # "충족" 판정이면 종료, 아니면 Worker로 재실행
    if "충족 여부: 충족" in reflection:
        return "end"
    
    return "worker"

# TODO 완료 후 테스트
reflection_state = reflection_node(final_state)
if reflection_state['reflection'] is not None:
    print("=== Reflection Node 결과 ===")
    print(f"검토 결과:\n{reflection_state['reflection'][:800]}...")
else:
    print("=== 코드를 완성해주세요 ===")

=== Reflection Node 결과 ===
검토 결과:
## 검토 결과
- **충족 여부:** 부분 충족  
- **강점:**  
  1. **구조화된 작업 흐름** – 요청된 10단계를 그대로 따라가며 각 단계마다 핵심 목표를 명확히 제시했습니다.  
  2. **관광·맛집·예산**을 모두 포함한 **항목 정의**와 **예산 항목 구분**이 잘 되어 있습니다.  
  3. **대중교통 중심 동선**과 **예상 비용**을 제시해 실용성을 높였습니다.  
  4. **추가 팁**(할인 패스, 앱 추천)도 제공하여 사용자가 바로 활용할 수 있습니다.  

- **개선 필요:**  
  1. **구체적인 일일 예산 금액**이 제시되지 않았습니다.  
     - 숙박·교통·식사·입장료·기타 비용을 실제 KRW 금액으로 추산하면 사용자가 예산을 직관적으로 판단할 수 있습니다.  
  2. **맛집 리스트**가 지역·코스만 제시돼 구체적 식당명·메뉴·가격·예약 여부가 부족합니다.  
     - 실제 방문 가능한 식당 2~3곳을 선정하고, 영업시간·예약 필요 여부·평균 가격대·대표적인 메뉴를 함께 제공해야 합니다.  
  3. **관광지 상세 정보**(운영시간·입장료·예약 여부·추천 관람 포인트)가 미흡합니다.  
     - 주요 관광지 별로 2025년 기준 최신 정보를 정확히 기재하면 신뢰도가 높아집니다.  
  4. **일정 최적화** 과정에서 이동 거리·소요시간을 구체적인 **지하철 역·도착·도착 시간**까지 명시하면 실제 여행 시 혼란 없이 활용할 수 있습니다.  
  5. **예산 배분**이 “일일 예산”·“총 예산”만 언급하고, **비용 항목 별 상세 금액**...


In [16]:
# TODO: 성과 평가 함수 구현
# 아래 함수를 완성하세요.

import json
import re

def evaluate_multi_agent(state: AgentState) -> dict:
    """
    Multi-Agent 시스템의 성과를 평가합니다.
    
    Args:
        state: 현재 AgentState (user_request, plan, result, reflection 포함)
    
    Returns:
        dict: {"score": int, "feedback": str, "details": dict}
    """

    # 평가 프롬프트 구성
    eval_prompt = f"""
    당신은 Multi-Agent 시스템의 성과를 평가하는 평가자입니다.
    
    ## 평가 기준
    1. Planner 평가 (0~30점): 계획의 완성도, 단계 구성의 적절성
    2. Worker 평가 (0~40점): 실행 품질, 정보의 충실도와 정확성
    3. 전체 평가 (0~30점): 사용자 요청 충족도, 협업 효율성
    
    ## 입력 정보
    - 사용자 요청: {state['user_request']}
    - Planner의 계획: {state['plan']}
    - Worker의 결과: {state['result']}
    - Reflection 검토: {state.get('reflection', '없음')}
    
    ## 출력 형식
    반드시 아래 JSON 형식만 출력하세요. 다른 텍스트는 포함하지 마세요.
    {{"planner_score": <0~30>, "worker_score": <0~40>, "overall_score": <0~30>, "feedback": "<전체 피드백>"}}
    """

    response = llm.invoke(eval_prompt)
    content = response.content

    # JSON 파싱 (마크다운 코드 블록 처리)
    try:
        # 마크다운 코드 블록에서 JSON 추출 (```json ... ``` 또는 ``` ... ```)
        json_match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', content)
        if json_match:
            json_str = json_match.group(1)
        else:
            # 코드 블록이 없으면 중괄호로 시작하는 JSON 추출
            json_match = re.search(r'\{[\s\S]*\}', content)
            json_str = json_match.group(0) if json_match else content

        result = json.loads(json_str)

        # TODO: result를 이용하여 점수를 합산하고 결과를 반영하세요
        total_score = result['planner_score'] + result['worker_score'] + result['overall_score']   # TODO: 세 항목의 점수를 합산하세요

        return {
            "score": total_score,
            "feedback": result['feedback'],       # TODO: result의 feedback 값으로 교체하세요
            "details": {
                "planner": result['planner_score'],    # TODO: result의 planner_score 값으로 교체하세요
                "worker": result['worker_score'],     # TODO: result의 worker_score 값으로 교체하세요
                "overall": result['overall_score']     # TODO: result의 overall_score 값으로 교체하세요
            }
        }
    except Exception as e:
        return {"score": 0, "feedback": f"평가 실패: {str(e)}", "details": {}}

# 테스트
evaluation = evaluate_multi_agent(reflection_state)
if evaluation['feedback'] is not None:
    print("=== 성과 평가 결과 ===")
    print(f"종합 점수: {evaluation['score']}/100")
    print(f"세부 점수: Planner {evaluation['details'].get('planner', 0)}/30, Worker {evaluation['details'].get('worker', 0)}/40, 전체 {evaluation['details'].get('overall', 0)}/30")
    print(f"피드백: {evaluation['feedback']}")
else:
    print("=== 코드를 완성해주세요 ===")

=== 성과 평가 결과 ===
종합 점수: 91/100
세부 점수: Planner 28/30, Worker 35/40, 전체 28/30
피드백: 계획은 10단계 구조를 잘 따라가며 관광·맛집·예산을 포괄했으나, 구체적인 KRW 금액과 상세 식당·맛집 정보가 부족해 실용성이 떨어집니다. 관광지 운영시간·입장료·예약 여부는 명시했으나, 이동 거리·소요시간과 지하철 역·도착 시간, 예산 항목별 상세 금액이 누락되었습니다. 추가로 실제 방문 가능한 식당 2~3곳과 가격·메뉴·예약 정보를 제공하고, 일일 예산을 150,000~200,000 KRW 수준으로 구체화하면 완성도가 크게 향상될 것입니다.
